In [2]:
!pip install -q unsloth transformers datasets accelerate peft trl gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.0/56.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.4/67.4 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 129.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 428.0/428.0 kB 36.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 120.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

In [3]:
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from transformers import TrainingArguments
from trl import SFTTrainer
import gradio as gr

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [6]:
dataset = load_dataset("gretelai/synthetic_text_to_sql")

train_data = dataset["train"].select(range(10000))   # small subset
val_data = dataset["test"].select(range(1000))

In [7]:
def format_prompt(example):
    return {
        "text": f"""### Question:
{example['sql_prompt']}

### Schema:
{example['sql_context']}

### SQL:
{example['sql']}"""
    }

train_data = train_data.map(format_prompt)
val_data = val_data.map(format_prompt)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [8]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/198 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.


In [9]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "v_proj"],
    lora_alpha = 16,
    lora_dropout = 0.1,
)

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.5.2 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [10]:
training_args = TrainingArguments(
    output_dir="./sql-model",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=20,
    fp16=True,
)

In [11]:
trainer = SFTTrainer(
    model=model,
    train_dataset=train_data,
    eval_dataset=val_data,
    dataset_text_field="text",
    max_seq_length=2048,
    tokenizer=tokenizer,
    args=training_args,
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
trainer.train()
trainer.save_model("sql-model")

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 1,250
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 6,815,744 of 8,037,076,992 (0.08% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
20,0.876018
40,0.673448
60,0.661399
80,0.620153
100,0.595806
120,0.601136
140,0.583379
160,0.559071
180,0.570698
200,0.571107


Unsloth: Restored added_tokens_decoder metadata in ./sql-model/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in ./sql-model/checkpoint-1000/tokenizer_config.json.


Step,Training Loss
20,0.876018
40,0.673448
60,0.661399
80,0.620153
100,0.595806
120,0.601136
140,0.583379
160,0.559071
180,0.570698
200,0.571107


Unsloth: Restored added_tokens_decoder metadata in ./sql-model/checkpoint-1250/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in sql-model/tokenizer_config.json.


In [12]:
def generate_sql(question, schema):
    prompt = f"""### Question:
{question}

### Schema:
{schema}

### SQL:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        temperature=0.2,
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [13]:
def predict(q, s):
    return generate_sql(q, s)

demo = gr.Interface(
    fn=predict,
    inputs=[
        gr.Textbox(label="Question"),
        gr.Textbox(label="Schema"),
    ],
    outputs=gr.Textbox(label="SQL"),
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://565df5087be992e412.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
import sqlite3

def execute_sql(schema_sql, query):
    conn = sqlite3.connect(":memory:")
    cursor = conn.cursor()

    try:
        # Create schema
        for stmt in schema_sql.split(";"):
            if stmt.strip():
                cursor.execute(stmt)

        # Execute query
        cursor.execute(query)
        result = cursor.fetchall()

        return True, result

    except Exception as e:
        return False, str(e)

    finally:
        conn.close()

In [15]:
def evaluate_model(model, tokenizer, dataset, num_samples=50):
    correct = 0

    samples = dataset.select(range(num_samples))

    for sample in samples:
        question = sample["sql_prompt"]
        schema = sample["sql_context"]
        gold_sql = sample["sql"]

        # Generate prediction
        pred_sql = generate_sql(question, schema)

        # Execute both
        pred_ok, pred_res = execute_sql(schema, pred_sql)
        gold_ok, gold_res = execute_sql(schema, gold_sql)

        if pred_ok and gold_ok and pred_res == gold_res:
            correct += 1

    accuracy = correct / num_samples
    print(f"✅ Execution Accuracy: {accuracy:.2f}")

    return accuracy

In [ ]:
evaluate_model(model, tokenizer, dataset["test"])

Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

✅ Execution Accuracy: 0.00


0.0

In [16]:
samples = dataset["test"].select(range(5))

for sample in samples:
    question = sample["sql_prompt"]
    schema = sample["sql_context"]
    gold_sql = sample["sql"]

    pred_sql = generate_sql(question, schema)

    print("\n===================")
    print("QUESTION:\n", question)
    print("\nGOLD SQL:\n", gold_sql)
    print("\nPREDICTED SQL:\n", pred_sql)


Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/


QUESTION:
 What is the average explainability score of creative AI applications in 'Europe' and 'North America' in the 'creative_ai' table?

GOLD SQL:
 SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'North America');

PREDICTED SQL:
 ### Question:
What is the average explainability score of creative AI applications in 'Europe' and 'North America' in the 'creative_ai' table?

### Schema:
CREATE TABLE creative_ai (application_id INT, name TEXT, region TEXT, explainability_score FLOAT); INSERT INTO creative_ai (application_id, name, region, explainability_score) VALUES (1, 'ApplicationX', 'Europe', 0.87), (2, 'ApplicationY', 'North America', 0.91), (3, 'ApplicationZ', 'Europe', 0.84), (4, 'ApplicationAA', 'North America', 0.93), (5, 'ApplicationAB', 'Europe', 0.89);

### SQL: 
SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'North America');

### Result:
0.90



Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
 Delete all records of rural infrastructure projects in Indonesia that have a completion date before 2010.

GOLD SQL:
 DELETE FROM rural_infrastructure WHERE country = 'Indonesia' AND completion_date < '2010-01-01';

PREDICTED SQL:
 ### Question:
Delete all records of rural infrastructure projects in Indonesia that have a completion date before 2010.

### Schema:
CREATE TABLE rural_infrastructure (id INT, project_name TEXT, sector TEXT, country TEXT, completion_date DATE); INSERT INTO rural_infrastructure (id, project_name, sector, country, completion_date) VALUES (1, 'Water Supply Expansion', 'Infrastructure', 'Indonesia', '2008-05-15'), (2, 'Rural Electrification', 'Infrastructure', 'Indonesia', '2012-08-28'), (3, 'Transportation Improvement', 'Infrastructure', 'Indonesia', '2009-12-31');

### SQL: 
DELETE FROM rural_infrastructure WHERE completion_date < '2010-01-01';

### Result:
After running the SQL query, the table will be updated to only contain the record with id = 

/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
 How many accidents have been recorded for SpaceX and Blue Origin rocket launches?

GOLD SQL:
 SELECT launch_provider, COUNT(*) FROM Accidents GROUP BY launch_provider;

PREDICTED SQL:
 ### Question:
How many accidents have been recorded for SpaceX and Blue Origin rocket launches?

### Schema:
CREATE TABLE Accidents (id INT, launch_provider VARCHAR(255), year INT, description TEXT); INSERT INTO Accidents (id, launch_provider, year, description) VALUES (1, 'SpaceX', 2015, 'Falcon 9 explosion'), (2, 'Blue Origin', 2011, 'Propulsion system failure'), (3, 'SpaceX', 2016, 'Falcon 9 explosion');

### SQL: 
SELECT COUNT(*) FROM Accidents WHERE launch_provider = 'SpaceX' AND year = 2015;


Both `max_new_tokens` (=100) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
 What is the maximum quantity of seafood sold in a single transaction?

GOLD SQL:
 SELECT MAX(quantity) FROM sales;

PREDICTED SQL:
 ### Question:
What is the maximum quantity of seafood sold in a single transaction?

### Schema:
CREATE TABLE sales (id INT, location VARCHAR(20), quantity INT, price DECIMAL(5,2)); INSERT INTO sales (id, location, quantity, price) VALUES (1, 'Northeast', 50, 12.99), (2, 'Midwest', 75, 19.99), (3, 'West', 120, 14.49);

### SQL: 
SELECT MAX(quantity) FROM sales;

### Output:
120


QUESTION:
 What is the total budget for movies released before 2010?

GOLD SQL:
 SELECT SUM(budget) FROM Movies_Release_Year WHERE release_year < 2010;

PREDICTED SQL:
 ### Question:
What is the total budget for movies released before 2010?

### Schema:
CREATE TABLE Movies_Release_Year (id INT, title VARCHAR(100), release_year INT, budget DECIMAL(10,2)); INSERT INTO Movies_Release_Year (id, title, release_year, budget) VALUES (1, 'The Matrix', 1999, 63000000.00), (2, '

In [17]:
def generate_sql(question, schema):
    prompt = f"""### Question:
{question}

### Schema:
{schema}

### SQL:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.1,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

    response = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Extract only SQL part
    if "### SQL:" in response:
        response = response.split("### SQL:")[-1]

    # Stop at semicolon
    if ";" in response:
        response = response.split(";")[0] + ";"

    return response.strip()


In [20]:
question = "Show all employee names"

schema = """
CREATE TABLE employees (
    id INT,
    name TEXT,
    salary INT
);
"""

result = generate_sql(question, schema)

print("Generated SQL:")
print(result)

Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated SQL:
SELECT name FROM employees;


In [23]:
def keyword_similarity(pred, gold):
    pred_words = set(pred.lower().split())
    gold_words = set(gold.lower().split())

    overlap = len(pred_words & gold_words)
    total = len(pred_words | gold_words)

    return overlap / total



score = keyword_similarity(pred, gold)



In [24]:
sample = dataset["test"][0]

pred = generate_sql(
    sample["sql_prompt"],
    sample["sql_context"]
)

gold = sample["sql"]

score = keyword_similarity(pred, gold)

print("Predicted SQL:")
print(pred)

print("\nGold SQL:")
print(gold)

print("\nSimilarity Score:")
print(score)

Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Predicted SQL:
SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'North America');

Gold SQL:
SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'North America');

Similarity Score:
1.0


In [25]:
def semantic_evaluation(dataset, num_samples=10):

    samples = dataset.select(range(num_samples))

    scores = []

    for sample in samples:

        pred = generate_sql(
            sample["sql_prompt"],
            sample["sql_context"]
        )

        gold = sample["sql"]

        score = keyword_similarity(pred, gold)

        scores.append(score)

        print("\n====================")
        print("QUESTION:")
        print(sample["sql_prompt"])

        print("\nPREDICTED:")
        print(pred)

        print("\nGOLD:")
        print(gold)

        print("\nSIMILARITY:")
        print(round(score, 2))

    avg_score = sum(scores) / len(scores)

    print("\n✅ Average Semantic Similarity:", round(avg_score, 2))

In [26]:
def simple_accuracy(dataset, num_samples=20):
    correct = 0

    samples = dataset.select(range(num_samples))

    for sample in samples:
        pred = generate_sql(
            sample["sql_prompt"],
            sample["sql_context"]
        )

        gold = sample["sql"]

        if pred.lower().strip() == gold.lower().strip():
            correct += 1

    acc = correct / num_samples
    print("Accuracy:", acc)



In [27]:
semantic_evaluation(dataset["test"], num_samples=5)

Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
What is the average explainability score of creative AI applications in 'Europe' and 'North America' in the 'creative_ai' table?

PREDICTED:
SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'North America');

GOLD:
SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'North America');

SIMILARITY:
1.0


Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
Delete all records of rural infrastructure projects in Indonesia that have a completion date before 2010.

PREDICTED:
DELETE FROM rural_infrastructure WHERE completion_date < '2010-01-01';

GOLD:
DELETE FROM rural_infrastructure WHERE country = 'Indonesia' AND completion_date < '2010-01-01';

SIMILARITY:
0.64


Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
How many accidents have been recorded for SpaceX and Blue Origin rocket launches?

PREDICTED:
SELECT COUNT(*) FROM Accidents WHERE launch_provider = 'SpaceX' AND year = 2015;

GOLD:
SELECT launch_provider, COUNT(*) FROM Accidents GROUP BY launch_provider;

SIMILARITY:
0.27


Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
What is the maximum quantity of seafood sold in a single transaction?

PREDICTED:
SELECT MAX(quantity) FROM sales;

GOLD:
SELECT MAX(quantity) FROM sales;

SIMILARITY:
1.0

QUESTION:
What is the total budget for movies released before 2010?

PREDICTED:
SELECT SUM(budget) FROM Movies_Release_Year WHERE release_year < 2010;

GOLD:
SELECT SUM(budget) FROM Movies_Release_Year WHERE release_year < 2010;

SIMILARITY:
1.0

✅ Average Semantic Similarity: 0.78


In [49]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [28]:
def exact_match_accuracy(dataset, num_samples=20):

    correct = 0

    samples = dataset.select(range(num_samples))

    for sample in samples:

        pred_sql = generate_sql(
            sample["sql_prompt"],
            sample["sql_context"]
        )

        gold_sql = sample["sql"]

        if pred_sql.lower().strip() == gold_sql.lower().strip():
            correct += 1

        print("\n====================")
        print("QUESTION:")
        print(sample["sql_prompt"])

        print("\nPREDICTED:")
        print(pred_sql)

        print("\nGOLD:")
        print(gold_sql)

    accuracy = correct / num_samples

    print(f"\n✅ Exact Match Accuracy: {accuracy:.2f}")

    return accuracy






In [31]:
exact_match_accuracy(dataset["test"], num_samples=5)

Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=80)


QUESTION:
What is the average explainability score of creative AI applications in 'Europe' and 'North America' in the 'creative_ai' table?

PREDICTED:
SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'North America');

GOLD:
SELECT AVG(explainability_score) FROM creative_ai WHERE region IN ('Europe', 'North America');


Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
Delete all records of rural infrastructure projects in Indonesia that have a completion date before 2010.

PREDICTED:
DELETE FROM rural_infrastructure WHERE completion_date < '2010-01-01';

GOLD:
DELETE FROM rural_infrastructure WHERE country = 'Indonesia' AND completion_date < '2010-01-01';


Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
How many accidents have been recorded for SpaceX and Blue Origin rocket launches?

PREDICTED:
SELECT COUNT(*) FROM Accidents WHERE launch_provider = 'SpaceX' AND year = 2015;

GOLD:
SELECT launch_provider, COUNT(*) FROM Accidents GROUP BY launch_provider;


Both `max_new_tokens` (=80) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



QUESTION:
What is the maximum quantity of seafood sold in a single transaction?

PREDICTED:
SELECT MAX(quantity) FROM sales;

GOLD:
SELECT MAX(quantity) FROM sales;

QUESTION:
What is the total budget for movies released before 2010?

PREDICTED:
SELECT SUM(budget) FROM Movies_Release_Year WHERE release_year < 2010;

GOLD:
SELECT SUM(budget) FROM Movies_Release_Year WHERE release_year < 2010;

✅ Exact Match Accuracy: 0.60


0.6